# FM Comparison — analysis & plots

All post-hoc analysis on the results of `fm_comparison.ipynb` (classification) and `fm_comparison_seg.ipynb` (segmentation). Loads the two `all_results.csv` files written by those execution notebooks; everything below is fast and reads from disk only.

Sections:
1. Setup — imports, paths, display names, color map
2. Classification — aggregation, results table, plots, rankings, bump plots
3. Segmentation — aggregation, results table, plots, rankings, bump plots
4. Bootstrap rank test — per-fold resampling, win-probability matrices, annotated bump plots (stars at statistically supported crossings)

## Setup

In [ ]:
from __future__ import annotations

from functools import reduce
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.lines import Line2D

### Load results

Reads the two `all_results.csv` files produced by the execution notebooks. Backbone / dataset / probe lists are auto-derived from the loaded data.

In [ ]:
CLS_OUTPUT_DIR = Path('eval_out/fm_comparison/')
SEG_OUTPUT_DIR = Path('eval_out/fm_comparison_seg/')
CLS_CSV = CLS_OUTPUT_DIR / 'all_results.csv'
SEG_CSV = SEG_OUTPUT_DIR / 'all_results.csv'

all_df  = pd.read_csv(CLS_CSV)
df_cls  = all_df                # alias for the bootstrap-rank section below
df_seg  = pd.read_csv(SEG_CSV)
print(f'cls rows: {len(all_df):,}   seg rows: {len(df_seg):,}')

# Auto-derive what to plot from what's actually in the CSVs.
_CLS_PROBE_ORDER = ['linear', 'knn', 'svm', 'prototype', 'zero_shot', 'finetune']
_SEG_PROBE_ORDER = ['linear_seg', 'conv_probe_seg', 'upernet_seg']
_CLS_DS_ORDER    = ['vindr_cxr', 'taix_ray']
_SEG_DS_ORDER    = ['siim_acr_ptx', 'montgomery_cxr']
_BB_ORDER = [
    'rad_dino', 'dinov3', 'chexfound', 'ark_plus', 'medical_mae', 'eva_x',
    'biomed_clip', 'chexagent', 'medsiglip', 'medimageinsights', 'siglip2',
]
BACKBONES_TO_RUN     = [b for b in _BB_ORDER if b in set(all_df['backbone']) | set(df_seg['backbone'])]
CLS_PROBES_TO_RUN    = [p for p in _CLS_PROBE_ORDER if p in set(all_df['evaluator'])]
SEG_PROBES_TO_RUN    = [p for p in _SEG_PROBE_ORDER if p in set(df_seg['evaluator'])]

# Restrict to datasets where every backbone has been evaluated. Otherwise the
# cross-dataset bump charts hit NaN ranks.
_cls_bb_count = all_df.groupby('dataset')['backbone'].nunique()
_seg_bb_count = df_seg.groupby('dataset')['backbone'].nunique()
_n_bb = len(BACKBONES_TO_RUN)
CLS_DATASETS_TO_RUN  = [d for d in _CLS_DS_ORDER if _cls_bb_count.get(d, 0) == _n_bb]
SEG_DATASETS_TO_RUN  = [d for d in _SEG_DS_ORDER if _seg_bb_count.get(d, 0) == _n_bb]

print('cls probes:',    CLS_PROBES_TO_RUN)
print('cls datasets:',  CLS_DATASETS_TO_RUN)
print('seg probes:',    SEG_PROBES_TO_RUN)
print('seg datasets:',  SEG_DATASETS_TO_RUN)


### Display names + colors

Underlying tables keep raw keys (`rad_dino`, `vindr_cxr`, …); display names are applied only at the plot/table layer via `dn(...)`.

In [ ]:
DISPLAY_NAMES = {
    # backbones
    'rad_dino': 'RAD-DINO', 'dinov3': 'DINOv3', 'chexfound': 'CheXFound',
    'ark_plus': 'Ark+', 'medical_mae': 'Medical MAE', 'eva_x': 'EVA-X',
    'biomed_clip': 'BiomedCLIP', 'chexagent': 'CheXagent',
    'medsiglip': 'MedSigLIP', 'medimageinsights': 'MedImageInsights',
    'siglip2': 'SigLIP 2',
    # datasets
    'vindr_cxr': 'VinDr-CXR', 'taix_ray': 'TAIX-Ray',
    'siim_acr_ptx': 'SIIM-ACR-PTX', 'montgomery_cxr': 'Montgomery',
    # probes (cls)
    'linear': 'Linear probe', 'knn': 'k-NN probe', 'svm': 'SVM probe',
    'prototype': 'Prototype', 'zero_shot': 'Zero-shot', 'finetune': 'Fine-tune',
    # probes (seg)
    'linear_seg': 'Linear (seg)', 'conv_probe_seg': 'Conv probe (seg)',
    'upernet_seg': 'UPerNet',
    # metrics
    'auroc': 'AUROC', 'auprc': 'AUPRC',
    'dice': 'Dice', 'iou': 'IoU', 'pixel_acc': 'Pixel Acc',
    'tpr': 'TPR', 'tnr': 'TNR', 'ppv': 'PPV', 'npv': 'NPV',
    'f1': 'F1', 'accuracy': 'Accuracy', 'balanced_accuracy': 'Balanced Acc',
    'mcc': 'MCC', 'threshold': 'Threshold',
}

def dn(x):
    return DISPLAY_NAMES.get(x, x)

_palette = plt.get_cmap('tab20').colors
BACKBONE_COLORS = {b: _palette[i % len(_palette)] for i, b in enumerate(BACKBONES_TO_RUN)}

def _fmt(mean, lo, hi):
    """Format a (mean, lo, hi) triple as 'mean [lo, hi]' (or 'mean' if no CI)."""
    if pd.isna(mean):
        return ""
    if pd.isna(lo) or pd.isna(hi):
        return f"{mean:.3f}"
    return f"{mean:.3f} [{lo:.3f}, {hi:.3f}]"

# Plot font sizes — uniform 11pt across every text element.
plt.rcParams.update({
    "font.size":           11,
    "axes.titlesize":      11,
    "axes.labelsize":      11,
    "xtick.labelsize":     11,
    "ytick.labelsize":     11,
    "legend.fontsize":     11,
    "legend.title_fontsize": 11,
    "figure.titlesize":    11,
})


## Classification — VinDr-CXR + TAIX-Ray

Aggregation, per-probe plots, ranking tables, and bump charts for the classification benchmark.

In [ ]:
# Bind classification probe/dataset names + OUTPUT_DIR for the cells below.
PROBES_TO_RUN   = CLS_PROBES_TO_RUN
DATASETS_TO_RUN = CLS_DATASETS_TO_RUN
OUTPUT_DIR      = CLS_OUTPUT_DIR

# Aggregate to (mean, lo, hi) per (dataset, evaluator, backbone, label) for each metric.
# CI source depends on which protocol produced the rows:
#   VinDr fixed-split: pool the bootstrap rows (bootstrap >= 0)  → 2.5 / 97.5 percentile.
#   TAIX-Ray k-fold:   normal-approx 95% CI of the mean from fold std:
#                          mean ± 1.96 · σ / √n_folds
#                      (n=5; we use the normal approximation per request —
#                       a t-dist with 4 df would give 2.776× and be a bit wider.)
METRICS = ["auroc", "auprc"]

# y-axis hints per metric — AUROC saturates near 1; AUPRC ranges widely.
_METRIC_LIMS = {
    "auroc": {"ylim": (0.5, 1.0), "yticks": [0.7, 0.8, 0.9, 1.0]},
    "auprc": {"ylim": (0.0, 1.0), "yticks": [0.0, 0.25, 0.5, 0.75, 1.0]},
}

def _agg(df, metric):
    s = df[metric]
    mean = s.mean()
    has_bootstrap = "bootstrap" in df.columns and (df["bootstrap"] >= 0).any()
    if has_bootstrap:
        boot = df.loc[df["bootstrap"] >= 0, metric]
        lo = boot.quantile(0.025)
        hi = boot.quantile(0.975)
    else:
        # k-fold: normal-approx 95% CI of the mean from fold std
        std = s.std(ddof=1)
        n_folds = int(s.notna().sum())
        half = 1.96 * std / np.sqrt(max(n_folds, 1))
        lo = mean - half
        hi = mean + half
    return pd.Series({"mean": mean, "lo": lo, "hi": hi})

aggs = {
    m: (all_df
        .groupby(["dataset", "evaluator", "backbone", "label"], dropna=False)
        .apply(lambda g, _m=m: _agg(g, _m), include_groups=False)
        .reset_index())
    for m in METRICS
}

# Per-(dataset, evaluator, backbone) macro: prefer existing 'macro_average' label rows,
# else mean over per-label rows (point estimate only, no CI for the fall-back).
def _build_macro(agg):
    if (agg["label"] == "macro_average").any():
        return agg[agg["label"] == "macro_average"].copy()
    return (
        agg[agg["label"] != "macro_average"]
        .groupby(["dataset", "evaluator", "backbone"], dropna=False)["mean"]
        .mean()
        .reset_index()
        .assign(lo=np.nan, hi=np.nan)
    )

macros = {m: _build_macro(a) for m, a in aggs.items()}

from functools import reduce

def _macro_with_label(macro_df):
    m = macro_df.copy()
    if "label" not in m.columns:
        m["label"] = "macro_average"
    return m[["dataset", "evaluator", "backbone", "label", "mean", "lo", "hi"]]

per_metric = {}
for metric in METRICS:
    per_label = aggs[metric][aggs[metric]["label"] != "macro_average"][
        ["dataset", "evaluator", "backbone", "label", "mean", "lo", "hi"]
    ]
    macro = _macro_with_label(macros[metric])
    rows = pd.concat([per_label, macro], ignore_index=True)
    rows[metric] = rows.apply(lambda r: _fmt(r["mean"], r["lo"], r["hi"]), axis=1)
    per_metric[metric] = rows[["dataset", "evaluator", "backbone", "label", metric]]

keys = ["dataset", "evaluator", "backbone", "label"]
results_table = reduce(
    lambda left, right: pd.merge(left, right, on=keys, how="outer"),
    [per_metric[m] for m in METRICS],
)

# Put macro_average at the top of each (dataset, evaluator, backbone) block.
results_table["_macro_first"] = (results_table["label"] != "macro_average").astype(int)
results_table = (
    results_table
    .sort_values(["dataset", "evaluator", "backbone", "_macro_first", "label"])
    .drop(columns="_macro_first")
    .reset_index(drop=True)
)

for _col in ["dataset", "evaluator", "backbone"]:
    results_table[_col] = results_table[_col].map(dn)
results_table = results_table.rename(columns=dn)
results_table.to_csv(OUTPUT_DIR / "all_results_table.csv", index=False)
print(f"Wrote {len(results_table):,} rows to {OUTPUT_DIR / 'all_results_table.csv'}")
results_table


### Plots

Five views of `aggs` / `macros`, all rendered once per metric in `METRICS`:

1. **Per-(probe, dataset) heatmap** — backbone × label, mean score.
2. **Per-(probe, dataset) radar** — same content as the heatmap, one
   polygon per backbone, labels on the spokes.
3. **Macro score per probe**, faceted by dataset — bars per backbone
   with CI whiskers.
4. **Macro score per backbone**, faceted by probe — paired bars across
   datasets, so you can read off cross-dataset transfer at a glance.
5. **Macro radar across probes** — one polygon per backbone, spokes are
   probes; faceted by dataset.

Plots 1–2 are per-label; plots 3–5 are macro. Heatmaps don't render CIs;
all bar / radar views do where they're available.


In [ ]:
# Plot 1: per-(metric × probe × dataset) heatmap — backbone × label score,
# plus a 'macro_average' column on the right for at-a-glance comparison.
# Colorbar range = data min/max so contrast adapts per (metric × probe × dataset).
# (mean only; heatmaps don't show CIs.)
for metric in METRICS:
    for probe in PROBES_TO_RUN:
        for ds in DATASETS_TO_RUN:
            sub = aggs[metric][(aggs[metric]["evaluator"] == probe)
                               & (aggs[metric]["dataset"] == ds)
                               & (aggs[metric]["label"] != "macro_average")]
            if sub.empty:
                continue
            pivot = sub.pivot(index="backbone", columns="label", values="mean")
            macro_col = (macros[metric][(macros[metric]["evaluator"] == probe)
                                        & (macros[metric]["dataset"] == ds)]
                         .set_index("backbone")["mean"])
            pivot["macro_average"] = macro_col.reindex(pivot.index)
            pivot = pivot.rename(index=dn)
            vmin = pivot.min(skipna=True).min()
            vmax = pivot.max(skipna=True).max()
            fig, ax = plt.subplots(figsize=(max(8, pivot.shape[1] * 0.6),
                                            max(4, pivot.shape[0] * 0.4)))
            sns.heatmap(pivot, annot=True, fmt=".2f", cmap="viridis", annot_kws={"fontsize": 11},
                        vmin=vmin, vmax=vmax, ax=ax)
            # White vertical line separating per-label cols from the macro col.
            ax.axvline(x=pivot.shape[1] - 1, color="white", linewidth=2)
            ax.set_title(f"{dn(probe)} — {dn(ds)} — mean {dn(metric)}")
            plt.tight_layout()
            plt.show()


In [ ]:
# Plot 1b: per-(metric × probe × dataset) radar — backbone × label, one line per backbone.
# Same content as Plot 1 (the heatmap) re-cast as overlaid radars, so per-label
# strengths and weaknesses across backbones are visible at a glance.
from math import pi

for metric in METRICS:
    lims = _METRIC_LIMS[metric]
    for probe in PROBES_TO_RUN:
        for ds in DATASETS_TO_RUN:
            sub = aggs[metric][(aggs[metric]["evaluator"] == probe)
                               & (aggs[metric]["dataset"] == ds)
                               & (aggs[metric]["label"] != "macro_average")]
            if sub.empty:
                continue
            pivot = sub.pivot(index="backbone", columns="label", values="mean")
            labels = list(pivot.columns)
            n_labels = len(labels)
            if n_labels < 3:
                print(f"radar skipped for {metric}/{probe}/{ds}: need >=3 labels, got {n_labels}")
                continue

            angles = [i * 2 * pi / n_labels for i in range(n_labels)]
            angles_closed = angles + [angles[0]]

            fig, ax = plt.subplots(figsize=(12, 9), subplot_kw=dict(polar=True))
            for backbone, row in pivot.iterrows():
                color = BACKBONE_COLORS[backbone]
                values = row.values.tolist()
                values_closed = values + [values[0]]
                ax.plot(angles_closed, values_closed, label=dn(backbone),
                        color=color, linewidth=1.6, marker="o", markersize=3)
                ax.fill(angles_closed, values_closed, color=color, alpha=0.04)

            ax.set_xticks(angles)
            ax.set_xticklabels(labels, fontsize=13)
            ax.set_ylim(*lims["ylim"])
            ax.set_yticks(lims["yticks"])
            ax.tick_params(axis="y", labelsize=13)
            ax.set_rlabel_position(180 / n_labels)
            ax.set_title(f"{dn(probe)} — {dn(ds)} — per-label {dn(metric)}", pad=20, fontsize=12)
            ax.legend(loc="upper right", bbox_to_anchor=(1.25, 1.05),
                      fontsize=12, title="backbone")
            plt.tight_layout()
            plt.show()


In [ ]:
# Plot 2: macro score per probe, faceted by (metric × dataset).
# Bars grouped by probe on the x-axis, one bar per backbone within each
# group, colored via BACKBONE_COLORS for consistency with Plots 1b / 4.
# Error-bar semantics differ by dataset:
#   VinDr fixed-split  → 95 % CI from bootstrap percentiles
#   TAIX-Ray k-fold    → 95 % CI of the mean, normal-approx from fold std
#                          (mean ± 1.96 · σ / √n_folds)
for metric in METRICS:
    macro_m = macros[metric]
    for ds in DATASETS_TO_RUN:
        sub = macro_m[macro_m["dataset"] == ds]
        if sub.empty:
            continue
        # Index = probe (group on x-axis); columns = backbone (bars within group).
        mean_p = sub.pivot(index="evaluator", columns="backbone", values="mean")
        lo_p   = sub.pivot(index="evaluator", columns="backbone", values="lo")
        hi_p   = sub.pivot(index="evaluator", columns="backbone", values="hi")
        # Preserve probe / backbone order from the *_TO_RUN lists.
        probe_order    = [p for p in PROBES_TO_RUN    if p in mean_p.index]
        backbone_order = [b for b in BACKBONES_TO_RUN if b in mean_p.columns]
        mean_p = mean_p.loc[probe_order, backbone_order]
        lo_p   = lo_p.loc[probe_order,   backbone_order]
        hi_p   = hi_p.loc[probe_order,   backbone_order]
        n_probes    = len(mean_p.index)
        n_backbones = len(mean_p.columns)
        x = np.arange(n_probes)
        width = 0.8 / max(n_backbones, 1)
        fig, ax = plt.subplots(figsize=(max(10, n_probes * max(n_backbones, 1) * 0.5), 5))
        for i, bb in enumerate(mean_p.columns):
            offset = (i - (n_backbones - 1) / 2) * width
            ax.bar(x + offset, mean_p[bb].values, width,
                   yerr=[(mean_p[bb] - lo_p[bb]).values,
                         (hi_p[bb]   - mean_p[bb]).values],
                   label=dn(bb), color=BACKBONE_COLORS.get(bb), capsize=3)
        err_label = "95 % CI (bootstrap)" if ds == "vindr_cxr" else "95 % CI (normal-approx, k-fold)"
        ax.set_xticks(x)
        ax.set_xticklabels([dn(p) for p in mean_p.index], rotation=30, ha="right")
        ax.set_ylabel(f"{dn(metric)} (macro, {err_label})")
        ax.set_title(f"Macro {dn(metric)} per probe — {dn(ds)} ({err_label})")
        ax.legend(title="backbone", bbox_to_anchor=(1.02, 1), loc="upper left")
        plt.tight_layout()
        plt.show()


In [ ]:
# Tabular companion to Plot 2: one table per probe.
# Each table — rows: backbones (alphabetic), columns: (metric, dataset),
# cells: 'mean [lo, hi]'. Saved as macro_table_<probe>.csv.
# Reuses _fmt() from the results-table cell.
records = []
for metric in METRICS:
    m = macros[metric][["dataset", "evaluator", "backbone", "mean", "lo", "hi"]].copy()
    m["metric"] = metric
    records.append(m)
macro_long = pd.concat(records, ignore_index=True)
macro_long["cell"] = macro_long.apply(
    lambda r: _fmt(r["mean"], r["lo"], r["hi"]), axis=1
)

macro_tables = {}
for probe in PROBES_TO_RUN:
    sub = macro_long[macro_long["evaluator"] == probe]
    if sub.empty:
        continue
    tbl = (
        sub.pivot(index="backbone",
                  columns=["metric", "dataset"],
                  values="cell")
        .sort_index()                       # backbones alphabetic
        .sort_index(axis=1, level=[0, 1])   # (metric, dataset) sorted
        .rename(index=dn, columns=dn)
    )
    out_csv = OUTPUT_DIR / f"macro_table_{probe}.csv"
    tbl.to_csv(out_csv)
    macro_tables[probe] = tbl
    print(f"=== {dn(probe)}  ({tbl.shape[0]} × {tbl.shape[1]})  →  {out_csv} ===")
    display(tbl)


In [ ]:
# Plot 4: radar — macro score per probe, one line per backbone, faceted by (metric × dataset).
# Spokes = probes; each backbone is overlaid as a closed polygon.
from math import pi

for metric in METRICS:
    lims = _METRIC_LIMS[metric]
    macro_m = macros[metric]
    for ds in DATASETS_TO_RUN:
        sub = macro_m[macro_m["dataset"] == ds]
        if sub.empty:
            continue
        pivot = sub.pivot(index="backbone", columns="evaluator", values="mean")
        # Keep probe order consistent with PROBES_TO_RUN; drop any missing
        probes_present = [p for p in PROBES_TO_RUN if p in pivot.columns]
        pivot = pivot[probes_present]
        n_probes = len(probes_present)
        if n_probes < 3:
            print(f"radar skipped for {metric}/{ds}: need >=3 probes, got {n_probes}")
            continue

        angles = [i * 2 * pi / n_probes for i in range(n_probes)]
        angles_closed = angles + [angles[0]]

        fig, ax = plt.subplots(figsize=(12, 9), subplot_kw=dict(polar=True))
        for backbone, row in pivot.iterrows():
            color = BACKBONE_COLORS[backbone]
            values = row.values.tolist()
            values_closed = values + [values[0]]
            ax.plot(angles_closed, values_closed, label=dn(backbone),
                    color=color, linewidth=1.8, marker="o", markersize=4)
            ax.fill(angles_closed, values_closed, color=color, alpha=0.05)

        ax.set_xticks(angles)
        ax.set_xticklabels([dn(p) for p in probes_present], fontsize=12)
        ax.set_ylim(*lims["ylim"])
        ax.set_yticks(lims["yticks"])
        ax.tick_params(axis="y", labelsize=12)
        ax.set_rlabel_position(180 / n_probes)
        ax.set_title(f"Macro {dn(metric)} per probe — {dn(ds)}", pad=20, fontsize=12)
        ax.legend(loc="upper right", bbox_to_anchor=(1.25, 1.05), fontsize=12, title="backbone")
        plt.tight_layout()
        plt.show()


### Rankings

Per `(metric, dataset)` ranking table — backbones sorted by mean macro score
across probes, with the per-probe macro alongside for context. Bump charts
of model order have been moved to the bootstrap-rank section below, where
each crossing is annotated with whether the flip is statistically supported.


In [ ]:
# Ranking table: per (metric × dataset), mean across probes + per-probe score.
for metric in METRICS:
    macro_m = macros[metric]
    for ds in DATASETS_TO_RUN:
        sub = macro_m[macro_m["dataset"] == ds]
        if sub.empty:
            continue
        pivot = sub.pivot(index="backbone", columns="evaluator", values="mean")
        probes_present = [p for p in PROBES_TO_RUN if p in pivot.columns]
        pivot = pivot[probes_present]
        pivot["mean_across_probes"] = pivot.mean(axis=1)
        pivot = pivot.sort_values("mean_across_probes", ascending=False)
        pivot.insert(0, "rank", range(1, len(pivot) + 1))
        print(f"\n=== {dn(ds)} — ranked by mean macro {dn(metric)} across probes ===")
        print(pivot.rename(index=dn, columns=dn).round(4).to_string())


## Segmentation — SIIM-ACR + Montgomery

Same structure as the classification section: aggregation, per-probe plots, ranking tables, and bump charts on foreground Dice.

In [ ]:
# Re-bind probe/dataset names + OUTPUT_DIR for the seg cells.
# The migrated seg cells expect `all_df` to be the segmentation frame (that's
# what the original seg notebook called it); alias here.
PROBES_TO_RUN    = SEG_PROBES_TO_RUN
DATASETS_TO_RUN  = SEG_DATASETS_TO_RUN
OUTPUT_DIR       = SEG_OUTPUT_DIR
all_df           = df_seg

# Foreground-only rows: keep label='macro_average' (background excluded by the evaluator's macro_average()).
METRIC_KEYS = ["dice", "iou"]

macro_rows = all_df[all_df["label"] == "macro_average"].copy()


def _agg_kfold(df, metric):
    vals = df[metric].to_numpy()
    mean = float(np.nanmean(vals))
    std = float(np.nanstd(vals, ddof=1)) if np.isfinite(vals).sum() > 1 else 0.0
    n = int(np.isfinite(vals).sum())
    half = 1.96 * std / np.sqrt(max(n, 1))
    return pd.Series({f"{metric}_mean": mean,
                      f"{metric}_lo":   mean - half,
                      f"{metric}_hi":   mean + half})

agg_rows = []
for (ds, ev, bb), sub in macro_rows.groupby(["dataset", "evaluator", "backbone"], dropna=False):
    row = {"dataset": ds, "evaluator": ev, "backbone": bb}
    for m in METRIC_KEYS:
        if m not in sub.columns:
            continue
        row.update(_agg_kfold(sub, m).to_dict())
    agg_rows.append(row)

agg = pd.DataFrame(agg_rows)

# Tidy results table: one row per (dataset, evaluator, backbone),
# one column per metric formatted as 'mean [lo, hi]'.
results_table = agg[["dataset", "evaluator", "backbone"]].copy()
for m in METRIC_KEYS:
    if f"{m}_mean" not in agg.columns:
        continue
    results_table[m] = agg.apply(
        lambda r, _m=m: _fmt(r[f"{_m}_mean"], r[f"{_m}_lo"], r[f"{_m}_hi"]),
        axis=1,
    )

results_table = (
    results_table
    .sort_values(["dataset", "evaluator", "backbone"])
    .reset_index(drop=True)
)
for _col in ["dataset", "evaluator", "backbone"]:
    results_table[_col] = results_table[_col].map(dn)
results_table = results_table.rename(columns=dn)
results_table.to_csv(OUTPUT_DIR / "all_results_table.csv", index=False)
print(f"Wrote {len(results_table):,} rows to {OUTPUT_DIR / 'all_results_table.csv'}")
results_table


### Plots

Two views of `agg`. Segmentation reports the foreground-only macro
(background excluded by the evaluator). Headline metric is **Dice**, with
**IoU** alongside.

1. **Per-(dataset, probe) heatmap** — backbone × metric (mean over folds),
   colorbar range driven by data min/max.
2. **Dice per probe**, faceted by dataset — bars grouped by probe on the
   x-axis, one bar per backbone within each group, colored via
   `BACKBONE_COLORS`. 95 % CI whiskers (normal-approx from fold std).

All error bars use the normal-approximation 95 % CI of the mean
(`mean ± 1.96 · σ / √n_folds`).


In [ ]:
# Plot 1: per-(dataset, evaluator) heatmap — backbone × metric (mean over
# folds, foreground-only macro). Colorbar range = data min/max so contrast
# adapts per (dataset × probe).
for ds in DATASETS_TO_RUN:
    for ev in PROBES_TO_RUN:
        sub = agg[(agg["dataset"] == ds) & (agg["evaluator"] == ev)]
        if sub.empty:
            continue
        mean_cols = [f"{m}_mean" for m in METRIC_KEYS if f"{m}_mean" in sub.columns]
        pivot = sub.set_index("backbone")[mean_cols]
        pivot.columns = [c.replace("_mean", "") for c in pivot.columns]
        pivot = pivot.sort_values("dice", ascending=False)
        pivot = pivot.rename(index=dn)
        vmin = pivot.min(skipna=True).min()
        vmax = pivot.max(skipna=True).max()
        fig, ax = plt.subplots(figsize=(max(8, pivot.shape[1] * 1.0),
                                        max(4, pivot.shape[0] * 0.4)))
        sns.heatmap(pivot, annot=True, fmt=".3f", cmap="viridis", annot_kws={"fontsize": 11},
                    vmin=vmin, vmax=vmax, ax=ax)
        ax.set_title(f"{dn(ev)} — {dn(ds)} — foreground macro (mean over folds)")
        plt.tight_layout()
        plt.show()


In [ ]:
# Plot 2: Dice per probe, faceted by dataset.
# Bars grouped by probe on the x-axis, one bar per backbone within each
# group, colored via BACKBONE_COLORS.
# Error bars: mean ± 1.96 · σ / √n_folds (normal-approx 95 % CI from fold std).
for ds in DATASETS_TO_RUN:
    sub = agg[agg["dataset"] == ds]
    if sub.empty:
        continue
    probe_order    = [p for p in PROBES_TO_RUN    if p in sub["evaluator"].unique()]
    backbone_order = [b for b in BACKBONES_TO_RUN if b in sub["backbone"].unique()]
    mean_p = sub.pivot(index="evaluator", columns="backbone", values="dice_mean").loc[probe_order, backbone_order]
    lo_p   = sub.pivot(index="evaluator", columns="backbone", values="dice_lo").loc[probe_order,   backbone_order]
    hi_p   = sub.pivot(index="evaluator", columns="backbone", values="dice_hi").loc[probe_order,   backbone_order]
    n_probes    = len(mean_p.index)
    n_backbones = len(mean_p.columns)
    x = np.arange(n_probes)
    width = 0.8 / max(n_backbones, 1)
    fig, ax = plt.subplots(figsize=(max(10, n_probes * max(n_backbones, 1) * 0.5), 5))
    for i, bb in enumerate(mean_p.columns):
        offset = (i - (n_backbones - 1) / 2) * width
        ax.bar(x + offset, mean_p[bb].values, width,
               yerr=[(mean_p[bb] - lo_p[bb]).values,
                     (hi_p[bb]   - mean_p[bb]).values],
               label=dn(bb), color=BACKBONE_COLORS.get(bb), capsize=3)
    ax.set_xticks(x)
    ax.set_xticklabels([dn(p) for p in mean_p.index], rotation=30, ha="right")
    ax.set_ylabel("Dice (foreground macro, 95 % CI normal-approx from k-fold std)")
    ax.set_title(f"Dice per probe — {dn(ds)}")
    ax.legend(title="backbone", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.show()


In [ ]:
# Tabular companion to Plot 2: one table per probe.
# Rows: backbones (alphabetic), columns: (metric, dataset), cells: 'mean [lo, hi]'.
# Saved as macro_table_<probe>.csv. Reuses _fmt() from the results-table cell.
macro_long_rows = []
for _, r in agg.iterrows():
    for m in METRIC_KEYS:
        if f"{m}_mean" not in agg.columns:
            continue
        macro_long_rows.append({
            "dataset":   r["dataset"],
            "evaluator": r["evaluator"],
            "backbone":  r["backbone"],
            "metric":    m,
            "cell":      _fmt(r[f"{m}_mean"], r[f"{m}_lo"], r[f"{m}_hi"]),
        })
macro_long = pd.DataFrame(macro_long_rows)

macro_tables = {}
for probe in PROBES_TO_RUN:
    sub = macro_long[macro_long["evaluator"] == probe]
    if sub.empty:
        continue
    tbl = (
        sub.pivot(index="backbone",
                  columns=["metric", "dataset"],
                  values="cell")
        .sort_index()                       # backbones alphabetic
        .sort_index(axis=1, level=[0, 1])   # (metric, dataset) sorted
        .rename(index=dn, columns=dn)
    )
    out_csv = OUTPUT_DIR / f"macro_table_{probe}.csv"
    tbl.to_csv(out_csv)
    macro_tables[probe] = tbl
    print(f"=== {dn(probe)}  ({tbl.shape[0]} × {tbl.shape[1]})  →  {out_csv} ===")
    display(tbl)


### Rankings

Per `(dataset, probe)` ranking table — backbones sorted by mean foreground
Dice, with IoU alongside for context. Bump charts of model order have been
moved to the bootstrap-rank section below, where each crossing is annotated
with whether the flip is statistically supported.


In [ ]:
for ds in DATASETS_TO_RUN:
    for ev in PROBES_TO_RUN:
        sub = agg[(agg["dataset"] == ds) & (agg["evaluator"] == ev)].copy()
        if sub.empty:
            continue
        sub = sub.sort_values("dice_mean", ascending=False).reset_index(drop=True)
        table = sub[["backbone"] + [f"{m}_mean" for m in METRIC_KEYS if f"{m}_mean" in sub.columns]].copy()
        table.columns = ["backbone"] + [c.replace("_mean", "") for c in table.columns[1:]]
        table.insert(0, "rank", range(1, len(table) + 1))
        print(f"\n=== {dn(ds)} / {dn(ev)} — ranked by mean foreground Dice ===")
        table["backbone"] = table["backbone"].map(dn)
        table = table.rename(columns=dn)
        print(table.round(4).to_string(index=False))

## Bootstrap rank test

A bump plot shows backbone rankings flipping between settings. To check whether a given flip is real or noise from the few folds we have, we resample units (folds for k-fold splits; seeds for VinDr) with replacement B = 1,000 times and recompute ranks. This yields:

1. **Per-backbone rank with 95 % CI** — stripe of likely positions.
2. **Pairwise win probability** P(rank A < rank B).
3. **Cross-setting flip probability** P(A > B in setting 1 AND A < B in setting 2) — directly tests whether a flip on the bump plot is supported by the data.

Unit = seed for VinDr-CXR (fixed-split, bootstrap = -1 point estimates); fold otherwise. n = 5 everywhere, so CI widths are comparable across settings.

In [ ]:
# ─── Bootstrap configuration ──────────────────────────────────────────
N_BOOTSTRAP = 1_000
RNG_SEED    = 0

BOOT_CLS_PROBES   = ['linear', 'knn', 'prototype']
BOOT_SEG_PROBES   = ['linear_seg', 'conv_probe_seg', 'upernet_seg']
BOOT_CLS_DATASETS = ['vindr_cxr', 'taix_ray']
BOOT_SEG_DATASETS = ['siim_acr_ptx', 'montgomery_cxr']

# ─── Build (unit × backbone) score matrices per setting ───────────────
def score_matrix(df: pd.DataFrame, dataset: str, evaluator: str, metric: str,
                 label: str = "macro_average") -> tuple[np.ndarray, list[str]]:
    """Return (n_units, n_backbones) score matrix and backbone names.

    VinDr unit = seed (fixed-split bootstrap=-1 row); else unit = fold.
    Drops backbones with any missing unit so all columns share the same rows.
    """
    sub = df.query("dataset == @dataset and evaluator == @evaluator and label == @label").copy()
    if dataset == "vindr_cxr":
        sub = sub.query("bootstrap == -1")
        unit_col = "seed"
    else:
        unit_col = "fold"
    piv = sub.pivot_table(index=unit_col, columns="backbone", values=metric, aggfunc="first")
    piv = piv.dropna(axis=1, how="any")
    return piv.values, list(piv.columns)

SETTINGS: dict[str, tuple[np.ndarray, list[str]]] = {}
for ds in BOOT_CLS_DATASETS:
    for ev in BOOT_CLS_PROBES:
        key = f"{ds} / {ev} / AUROC"
        SETTINGS[key] = score_matrix(df_cls, ds, ev, metric="auroc")
for ds in BOOT_SEG_DATASETS:
    for ev in BOOT_SEG_PROBES:
        key = f"{ds} / {ev} / Dice"
        SETTINGS[key] = score_matrix(df_seg, ds, ev, metric="dice")

for k, (m, bbs) in SETTINGS.items():
    print(f"{k:50s}  units={m.shape[0]}  backbones={len(bbs)}")

# ─── Bootstrap the rank ───────────────────────────────────────────────
# For each of B rounds we resample the unit rows with replacement (one
# resample shared across all backbones, so within-fold paired structure is
# preserved), then re-rank by the resampled per-backbone mean.
def bootstrap_ranks(scores: np.ndarray, B: int = N_BOOTSTRAP,
                    rng: np.random.Generator | None = None,
                    higher_is_better: bool = True) -> np.ndarray:
    """Return (B, n_backbones) integer ranks (1 = best)."""
    rng = rng if rng is not None else np.random.default_rng(RNG_SEED)
    n_units = scores.shape[0]
    idx = rng.integers(0, n_units, size=(B, n_units))           # paired resample over units
    means = scores[idx].mean(axis=1)                            # (B, n_backbones)
    sign = -1.0 if higher_is_better else 1.0
    order = np.argsort(sign * means, axis=1, kind="stable")
    ranks = np.empty_like(order)
    np.put_along_axis(ranks, order, np.arange(1, scores.shape[1] + 1), axis=1)
    return ranks

RANK_DIST: dict[str, np.ndarray] = {}
BACKBONES: dict[str, list[str]] = {}
for key, (scores, bbs) in SETTINGS.items():
    RANK_DIST[key] = bootstrap_ranks(scores, B=N_BOOTSTRAP,
                                     rng=np.random.default_rng(RNG_SEED))
    BACKBONES[key] = bbs
print("Computed rank distributions for", len(RANK_DIST), "settings.")

### Per-backbone rank with 95 % CI


In [ ]:
def rank_summary(key: str) -> pd.DataFrame:
    ranks = RANK_DIST[key]
    bbs   = BACKBONES[key]
    rows = []
    for j, b in enumerate(bbs):
        r = ranks[:, j]
        rows.append({
            "backbone": b,
            "mean_rank": r.mean(),
            "median_rank": float(np.median(r)),
            "ci_lo": float(np.quantile(r, 0.025)),
            "ci_hi": float(np.quantile(r, 0.975)),
        })
    return pd.DataFrame(rows).sort_values("mean_rank").reset_index(drop=True)

for key in RANK_DIST:
    print("===", key)
    print(rank_summary(key).to_string(index=False))
    print()

# Visualize the rank CI for each setting as a strip plot.
n_settings = len(RANK_DIST)
ncols = 3
nrows = int(np.ceil(n_settings / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(5.2 * ncols, 0.42 * 11 * nrows), squeeze=False)
for ax, (key, ranks) in zip(axes.flat, RANK_DIST.items()):
    bbs = BACKBONES[key]
    summary = rank_summary(key)
    order = summary["backbone"].tolist()
    y = np.arange(len(order))
    lo = summary["ci_lo"].values
    hi = summary["ci_hi"].values
    mean = summary["mean_rank"].values
    ax.hlines(y, lo, hi, color="#4c72b0", lw=4, alpha=0.6)
    ax.scatter(mean, y, color="#1f3a68", zorder=5, s=30)
    ax.set_yticks(y)
    ax.set_yticklabels([dn(b) for b in order], fontsize=11)
    ax.invert_yaxis()
    ax.set_xlim(0.5, len(bbs) + 0.5)
    ax.set_xticks(range(1, len(bbs) + 1))
    ax.set_xlabel("rank (1 = best)")
    ax.set_title(" / ".join(dn(p) for p in key.split(" / ")), fontsize=11)
    ax.grid(axis="x", alpha=0.3)
for ax in axes.flat[n_settings:]:
    ax.axis("off")
fig.suptitle("Bootstrap rank with 95 % CI (B = {:,})".format(N_BOOTSTRAP), y=1.001)
fig.tight_layout()

### Pairwise win probability

For each setting, entry (i, j) is P(rank of backbone i < rank of backbone j). Cells ≥ 0.975 (row strictly beats column) and ≤ 0.025 are highlighted.

In [ ]:
def pairwise_winprob(key: str) -> pd.DataFrame:
    ranks = RANK_DIST[key]
    bbs   = BACKBONES[key]
    n = len(bbs)
    out = np.zeros((n, n))
    for i in range(n):
        out[i] = (ranks[:, [i]] < ranks).mean(axis=0)
    df = pd.DataFrame(out, index=bbs, columns=bbs)
    order = rank_summary(key)["backbone"].tolist()
    return df.loc[order, order]

def _plot_winprob_grid(probes, datasets, metric, suptitle):
    """Heatmap grid: rows = probes, cols = datasets, side by side."""
    nrows, ncols = len(probes), len(datasets)
    n_bb = max(len(BACKBONES[f"{ds} / {ev} / {metric}"])
               for ds in datasets for ev in probes)
    fig, axes = plt.subplots(
        nrows, ncols,
        figsize=(0.55 * n_bb * ncols + 2.5, 0.55 * n_bb * nrows + 1.0),
        squeeze=False,
    )
    for i, ev in enumerate(probes):
        for j, ds in enumerate(datasets):
            key = f"{ds} / {ev} / {metric}"
            wp = pairwise_winprob(key).rename(index=DISPLAY_NAMES, columns=DISPLAY_NAMES)
            ax = axes[i, j]
            sns.heatmap(
                wp, annot=True, fmt=".2f", cmap="RdBu_r", center=0.5,
                vmin=0, vmax=1, square=True,
                cbar=(j == ncols - 1),
                cbar_kws={"label": "P(row beats col)"} if j == ncols - 1 else None,
                annot_kws={"fontsize": 9}, ax=ax,
            )
            ax.set_title(f"{dn(ds)} — {dn(ev)}", fontsize=11)
    fig.suptitle(suptitle, fontsize=13)
    fig.tight_layout()

_plot_winprob_grid(BOOT_CLS_PROBES, BOOT_CLS_DATASETS, "AUROC",
                   "Pairwise win probability — Classification")
_plot_winprob_grid(BOOT_SEG_PROBES, BOOT_SEG_DATASETS, "Dice",
                   "Pairwise win probability — Segmentation")

# Tabular summary: count of significant wins (P >= 0.975) per backbone per setting.
rows = []
for key in RANK_DIST:
    wp = pairwise_winprob(key)
    n = len(wp)
    sig_wins  = (wp.values >= 0.975).sum(axis=1)
    sig_loss  = (wp.values <= 0.025).sum(axis=1)
    ties      = n - 1 - sig_wins - sig_loss
    for b, w, l, t in zip(wp.index, sig_wins, sig_loss, ties):
        rows.append({"setting": key, "backbone": b,
                     "sig_wins": int(w), "sig_losses": int(l), "ties": int(t)})
sig_df = pd.DataFrame(rows)
print(sig_df.pivot(index="backbone", columns="setting", values="sig_wins").fillna(0).astype(int))

### Cross-setting flip test

For two settings X and Y and two backbones A and B:

$$P_{\text{flip}} = P\bigl(\text{rank}_X(A) < \text{rank}_X(B) \;\land\; \text{rank}_Y(A) > \text{rank}_Y(B)\bigr).$$

If $P_{\text{flip}} \ge 0.95$ the bump on the plot is well-supported. Settings share the same bootstrap RNG seed, so each row of the rank matrix is the same resample world — the flip is computed jointly over the two settings, not independently.

In [ ]:
def flip_prob(key_x: str, key_y: str) -> pd.DataFrame:
    """P(A outranks B in X AND B outranks A in Y) for every ordered pair (A, B)."""
    rx, bx = RANK_DIST[key_x], BACKBONES[key_x]
    ry, by = RANK_DIST[key_y], BACKBONES[key_y]
    common = [b for b in bx if b in by]
    ix = [bx.index(b) for b in common]
    iy = [by.index(b) for b in common]
    rx, ry = rx[:, ix], ry[:, iy]
    n = len(common)
    out = np.zeros((n, n))
    for i in range(n):
        a_better_in_x = rx[:, [i]] < rx          # (B, n)
        a_worse_in_y  = ry[:, [i]] > ry
        out[i] = (a_better_in_x & a_worse_in_y).mean(axis=0)
    return pd.DataFrame(out, index=common, columns=common)


def significant_flips(key_x: str, key_y: str, threshold: float = 0.95) -> pd.Series:
    """(A, B) -> P_flip for ordered pairs with P_flip >= threshold (self-pairs dropped)."""
    stacked = flip_prob(key_x, key_y).stack()
    stacked = stacked[stacked.index.get_level_values(0)
                      != stacked.index.get_level_values(1)]
    return stacked[stacked >= threshold].sort_values(ascending=False)


# Example: flips between the three probe heads on SIIM-ACR.
siim_keys = [k for k in RANK_DIST if k.startswith("siim_acr_ptx")]
for i in range(len(siim_keys)):
    for j in range(i + 1, len(siim_keys)):
        kx, ky = siim_keys[i], siim_keys[j]
        sig = significant_flips(kx, ky, threshold=0.95)
        probe_x = kx.split(' / ')[1]
        probe_y = ky.split(' / ')[1]
        print(f"\n=== {kx}\n    vs {ky}")
        if sig.empty:
            print("  no pair with P_flip >= 0.95")
            continue
        for (a, b), p in sig.items():
            print(f"  {a:18s}  >  {b:18s}  in {probe_x:<16s}  |  reversed in {probe_y:<16s}  P_flip={p:.3f}")

# Same flip test, but across datasets for a single probe (e.g. UPerNet on SIIM vs. Montgomery).
pairs_to_check = [
    ("siim_acr_ptx / upernet_seg / Dice", "montgomery_cxr / upernet_seg / Dice"),
    ("vindr_cxr / linear / AUROC",        "taix_ray / linear / AUROC"),
    ("vindr_cxr / linear / AUROC",        "vindr_cxr / knn / AUROC"),
]
for kx, ky in pairs_to_check:
    if kx not in RANK_DIST or ky not in RANK_DIST:
        continue
    sig = significant_flips(kx, ky, threshold=0.95)
    print(f"\n=== {kx}\n    vs {ky}")
    if sig.empty:
        print("  no pair with P_flip >= 0.95")
        continue
    for (a, b), p in sig.items():
        print(f"  {a:18s}  >  {b:18s}  in setting 1   |   reversed in setting 2   P_flip={p:.3f}")

### How to read this against the bump plot

- **Strip plot (rank ± 95 % CI).** Narrow stripes = the rank is well-determined. Stripes that overlap across two backbones mean the ordering between those backbones at that setting is not significant.
- **Pairwise heatmap.** Cells ≥ 0.975 are the bilateral significant orderings. Read row-wise: how many backbones the row beats at the 0.975 level.
- **Flip test.** For every two settings on the bump plot, $P_{\text{flip}}\ge0.95$ for a (A, B) pair means the bump that crosses between A and B between those two settings is real. Pairs without significant flip are bumps that the underlying variance does not support — they may still be visible in the figure, but treat them as exploratory.

### Annotated bump plot

Draws the standard bump plot across a sequence of settings and adds a star marker at every line crossing whose flip probability passes the threshold. Unmarked crossings exist in the figure but the data does not support them.

In [ ]:
from matplotlib.lines import Line2D

SUBPLOT_W, SUBPLOT_H = 6.0, 6.0   # uniform per-subplot size for all bump figures


def _point_ranks(key):
    # Integer rank (1 = best) per backbone, from the per-unit mean score.
    scores, bbs = SETTINGS[key]
    means = scores.mean(axis=0)
    order = np.argsort(-means)
    ranks = np.empty(len(bbs), dtype=int)
    ranks[order] = np.arange(1, len(bbs) + 1)
    return dict(zip(bbs, ranks.tolist()))


def annotated_bump(setting_keys, threshold=0.95, title=None,
                   x_labels=None, figsize=None, palette="tab20", ax=None):
    common = list(BACKBONES[setting_keys[0]])
    for k in setting_keys[1:]:
        common = [b for b in common if b in BACKBONES[k]]

    rk = {k: _point_ranks(k) for k in setting_keys}
    mean_rank = {b: float(np.mean([rk[k][b] for k in setting_keys])) for b in common}
    common.sort(key=lambda b: mean_rank[b])

    n_settings  = len(setting_keys)
    n_backbones = len(common)
    x = np.arange(n_settings)

    if ax is None:
        fig, ax = plt.subplots(figsize=figsize or (max(8, n_settings * 2.5),
                                                    max(6, n_backbones * 0.4)))
    else:
        fig = ax.figure
    colors = {b: BACKBONE_COLORS[b] for b in common}

    for b in common:
        y = [rk[k][b] for k in setting_keys]
        ax.plot(x, y, marker="o", linewidth=1.8, markersize=7,
                color=colors[b], alpha=0.55)
        ax.annotate(dn(b), xy=(x[-1] + 0.05, y[-1]), va="center",
                    fontsize=11, color=colors[b])

    n_sig = 0
    for i in range(n_settings - 1):
        kx, ky = setting_keys[i], setting_keys[i + 1]
        fp = flip_prob(kx, ky)
        for ai in range(len(common)):
            for bi in range(ai + 1, len(common)):
                a, b = common[ai], common[bi]
                rax, rbx = rk[kx][a], rk[kx][b]
                ray, rby = rk[ky][a], rk[ky][b]
                if (rax - rbx) * (ray - rby) >= 0:
                    continue
                p = fp.loc[a, b] if rax < rbx else fp.loc[b, a]
                if p < threshold:
                    continue
                dx, dy = rbx - rax, rby - ray
                t = dx / (dx - dy)
                x_cross = i + t
                y_cross = rax + t * (ray - rax)
                ax.scatter(x_cross, y_cross, marker="*", s=220,
                           color="black", edgecolor="white",
                           linewidth=0.9, zorder=10)
                n_sig += 1

    ax.set_xticks(x)
    ax.set_xticklabels(x_labels if x_labels else setting_keys, rotation=0)
    ax.set_yticks(range(1, n_backbones + 1))
    ax.set_ylim(n_backbones + 0.5, 0.5)
    ax.set_ylabel("rank (1 = best)")
    ax.set_title(title if title else " -> ".join(setting_keys), fontsize=11)
    ax.grid(axis="y", alpha=0.3)
    ax.legend(handles=[Line2D([0], [0], marker="*", color="w",
                              markerfacecolor="black", markeredgecolor="white",
                              markersize=14,
                              label=f"P_flip >= {threshold} ({n_sig} crossings)")],
              loc="lower center", fontsize=11)
    return fig


# ── Cross-dataset bumps — classification, one column per probe ──────────
cls_probes = ["linear", "knn", "prototype"]
cls_probe_labels = {"linear": "Linear", "knn": "k-NN", "prototype": "Prototype"}
fig, axes = plt.subplots(1, len(cls_probes),
                         figsize=(SUBPLOT_W * len(cls_probes), SUBPLOT_H))
for ax, probe in zip(axes, cls_probes):
    annotated_bump(
        [f"vindr_cxr / {probe} / AUROC", f"taix_ray / {probe} / AUROC"],
        threshold=0.95,
        x_labels=["VinDr-CXR", "TAIX-Ray"],
        title=cls_probe_labels[probe],
        ax=ax,
    )
fig.suptitle("Cross-dataset bump — classification (significant crossings starred)",
             fontsize=13)
fig.tight_layout()

# ── Cross-dataset bumps — segmentation, one column per probe ────────────
seg_probes = ["linear_seg", "conv_probe_seg", "upernet_seg"]
seg_probe_labels = {"linear_seg": "Linear", "conv_probe_seg": "Conv",
                    "upernet_seg": "UPerNet"}
fig, axes = plt.subplots(1, len(seg_probes),
                         figsize=(SUBPLOT_W * len(seg_probes), SUBPLOT_H))
for ax, probe in zip(axes, seg_probes):
    annotated_bump(
        [f"siim_acr_ptx / {probe} / Dice", f"montgomery_cxr / {probe} / Dice"],
        threshold=0.95,
        x_labels=["SIIM-ACR", "Montgomery"],
        title=seg_probe_labels[probe],
        ax=ax,
    )
fig.suptitle("Cross-dataset bump — segmentation (significant crossings starred)",
             fontsize=13)
fig.tight_layout()

# ── Cross-probe bumps — classification, one column per dataset ──────────
cls_datasets = [("vindr_cxr", "VinDr-CXR"), ("taix_ray", "TAIX-Ray")]
fig, axes = plt.subplots(1, len(cls_datasets),
                         figsize=(SUBPLOT_W * len(cls_datasets), SUBPLOT_H))
for ax, (ds, x_lab) in zip(axes, cls_datasets):
    annotated_bump(
        [f"{ds} / linear / AUROC",
         f"{ds} / knn / AUROC",
         f"{ds} / prototype / AUROC"],
        threshold=0.95,
        x_labels=["Linear", "k-NN", "Prototype"],
        title=x_lab,
        ax=ax,
    )
fig.suptitle("Cross-probe bump — classification (significant crossings starred)",
             fontsize=13)
fig.tight_layout()

# ── Cross-probe bumps — segmentation, one column per dataset ────────────
seg_datasets = [("siim_acr_ptx", "SIIM-ACR"), ("montgomery_cxr", "Montgomery")]
fig, axes = plt.subplots(1, len(seg_datasets),
                         figsize=(SUBPLOT_W * len(seg_datasets), SUBPLOT_H))
for ax, (ds, x_lab) in zip(axes, seg_datasets):
    annotated_bump(
        [f"{ds} / linear_seg / Dice",
         f"{ds} / conv_probe_seg / Dice",
         f"{ds} / upernet_seg / Dice"],
        threshold=0.95,
        x_labels=["Linear", "Conv", "UPerNet"],
        title=x_lab,
        ax=ax,
    )
fig.suptitle("Cross-probe bump — segmentation (significant crossings starred)",
             fontsize=13)
fig.tight_layout()
plt.show()

In [ ]:
# Restructure all_results_table.csv into flat per-probe tables (CI already baked in).
# Source: eval_out/fm_comparison{,_seg}/all_results_table.csv — values are
# already "mean [lo, hi]" strings, display-named via dn().
# Output:
#   cls — one CSV per (probe, dataset, metric): rows=backbone, cols=label,
#         macro_average first then alphabetic.
#   seg — one CSV per (probe, dataset):         rows=backbone, cols=metric
#         (no `label` dim in the seg results table — it's already macro-only).

_cls_tbl = pd.read_csv(CLS_OUTPUT_DIR / "all_results_table.csv")
_seg_tbl = pd.read_csv(SEG_OUTPUT_DIR / "all_results_table.csv")

# ── classification ─────────────────────────────────────────────────────────
_cls_metric_cols = [c for c in _cls_tbl.columns
                    if c not in {"dataset", "evaluator", "backbone", "label"}]
flat_cls = {}
(CLS_OUTPUT_DIR / "flat").mkdir(parents=True, exist_ok=True)
for (probe, ds), sub in _cls_tbl.groupby(["evaluator", "dataset"], sort=False):
    for metric in _cls_metric_cols:
        tbl = (sub.pivot(index="backbone", columns="label", values=metric)
                  .sort_index())
        cols = list(tbl.columns)
        if "macro_average" in cols:
            cols = ["macro_average"] + sorted(c for c in cols if c != "macro_average")
        else:
            cols = sorted(cols)
        tbl = tbl[cols]
        flat_cls[(probe, ds, metric)] = tbl
        tbl.to_csv(CLS_OUTPUT_DIR / "flat" / f"{probe}_{ds}_{metric}.csv")

# ── segmentation ───────────────────────────────────────────────────────────
_seg_metric_cols = [c for c in _seg_tbl.columns
                    if c not in {"dataset", "evaluator", "backbone"}]
flat_seg = {}
(SEG_OUTPUT_DIR / "flat").mkdir(parents=True, exist_ok=True)
for (probe, ds), sub in _seg_tbl.groupby(["evaluator", "dataset"], sort=False):
    tbl = (sub.set_index("backbone")[_seg_metric_cols]
              .sort_index())
    flat_seg[(probe, ds)] = tbl
    tbl.to_csv(SEG_OUTPUT_DIR / "flat" / f"{probe}_{ds}.csv")

print(f"cls tables written: {len(flat_cls)}  →  {CLS_OUTPUT_DIR / 'flat'}")
print(f"seg tables written: {len(flat_seg)}  →  {SEG_OUTPUT_DIR / 'flat'}")

# Preview one of each.
for k in [("Linear probe", "VinDr-CXR", "AUROC")]:
    if k in flat_cls:
        print(f"\n=== cls {k} ===")
        display(flat_cls[k])
for k in [("Linear (seg)", "SIIM-ACR-PTX")]:
    if k in flat_seg:
        print(f"\n=== seg {k} ===")
        display(flat_seg[k])


In [ ]:
df_all_cls = pd.read_csv(CLS_OUTPUT_DIR / "all_results_table.csv")
df_macro_cls = df_all_cls[df_all_cls.label == "macro_average"]
df_macro_cls_taixray  = df_macro_cls[df_macro_cls.dataset == "TAIX-Ray"]
df_macro_cls_vindrcxr = df_macro_cls[df_macro_cls.dataset == "VinDr-CXR"]

df_all_seg = pd.read_csv(SEG_OUTPUT_DIR / "all_results_table.csv")
df_all_seg_montgomery = df_all_seg[df_all_seg.dataset == "Montgomery"]
df_all_seg_siimacrptx = df_all_seg[df_all_seg.dataset == "SIIM-ACR-PTX"]

# ── wide format: one table per (dataset, metric) — rows=backbone, cols=probe
_CLS_METRICS = ["AUROC", "AUPRC"]
_SEG_METRICS = ["Dice", "IoU"]

def _wide(df, metric):
    return (df.pivot(index="backbone", columns="evaluator", values=metric)
              .sort_index(axis=0)
              .sort_index(axis=1))

(CLS_OUTPUT_DIR / "tables").mkdir(parents=True, exist_ok=True)
(SEG_OUTPUT_DIR / "tables").mkdir(parents=True, exist_ok=True)

wide_cls = {}
for ds, df in [("TAIX-Ray", df_macro_cls_taixray),
               ("VinDr-CXR", df_macro_cls_vindrcxr)]:
    for m in _CLS_METRICS:
        tbl = _wide(df, m)
        wide_cls[(ds, m)] = tbl
        tbl.to_csv(CLS_OUTPUT_DIR / "tables" / f"{ds}_{m}.csv")

wide_seg = {}
for ds, df in [("Montgomery", df_all_seg_montgomery),
               ("SIIM-ACR-PTX", df_all_seg_siimacrptx)]:
    for m in _SEG_METRICS:
        tbl = _wide(df, m)
        wide_seg[(ds, m)] = tbl
        tbl.to_csv(SEG_OUTPUT_DIR / "tables" / f"{ds}_{m}.csv")

for k, t in wide_cls.items():
    print(f"\n=== cls {k} ===")
    display(t)
for k, t in wide_seg.items():
    print(f"\n=== seg {k} ===")
    display(t)
